# 01. The data: Ken French's daily and monthly factor files, downloaded and counted

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Factor | the monthly or daily return of one long-short portfolio that captures one source of common movement in stock returns. The six here: the market above cash (Mkt-RF), small minus big firms (SMB), cheap minus expensive firms (HML), past winners minus past losers (Mom), profitable minus unprofitable firms (RMW), and firms that invest little minus firms that invest a lot (CMA) |
| Risk-free rate | the return on a one-month US Treasury bill (RF in French's files) |
| Excess return | a return minus the risk-free rate over the same period. Mkt-RF is the market's return minus the risk-free rate; the other five factors are long-short portfolios, paid for by their own short side, so their returns are excess returns as well |
| Trading day | a day on which the New York Stock Exchange traded. French's daily files have one row per trading day |
| Realised variance | the variance of a factor computed from the daily returns of one month. It is the signal this study evaluates, built in notebook 02 |
| Value-weighted return | the return of a group of firms in which each firm counts in proportion to its market value at the start of the period |
| Decile | one tenth of the stocks, ranked by a characteristic. The ten momentum portfolios rank stocks by their return from twelve months to two months before the portfolio is formed |
| WML | winners minus losers: the top momentum decile's return minus the bottom decile's, the momentum series of Barroso and Santa-Clara (2015) |
| Vintage | the version of French's files on the download date, named by the release of the CRSP database they were built from (202609 is the September 2026 release) |
| Provenance | the record of what was downloaded, when, with its checksum and vintage |
| Sample | the months a comparison runs on. The study uses four, listed below |

**What this notebook does.** It downloads the eight files of Ken French's data library this study reads, records what it downloaded (the date, the address, the SHA-256 checksum of each zip and the CRSP vintage French prints at the top of each file), turns the files into tables of the six factors and of WML, monthly and daily, and counts what is in them: months, trading days, and the months each sample holds.

**Why it comes first.** Every number in the study is computed from these files, and French revises history: a return from 1985 can differ between the file served today and the file Moreira and Muir downloaded in 2015. A replication that misses a published number has to be able to say whether the code or the data moved. The provenance record written here makes that possible, and every later output carries it.

**Which expectations it judges.** None. The six expectations, E1 to E6, are fixed in `src/sq/constants.py` and judged in notebooks 03 to 08. This notebook checks the counts those expectations rest on.

**The papers.** Moreira and Muir (2017), "Volatility-Managed Portfolios", Journal of Finance 72(4), 1611-1644; Cederburg, O'Doherty, Wang and Yan (2020), "On the performance of volatility-managed portfolios", Journal of Financial Economics 138(1), 95-117; Barroso and Santa-Clara (2015), "Momentum has its moments", Journal of Financial Economics 116(1), 111-120. Page numbers below are the journals'.

**What I expect to see, and where it comes from.**

- Eight files, each with a vintage line. On 10 November 2026 the library served vintage 202609, which runs to September 2026; the loader cuts every file at August 2026, the end of the sample fixed in the constants (`SAMPLE_END`), so a later vintage gives the same months.
- Monthly rows to August 2026: **1,202** in the three-factor file (from July 1926), **758** in the five-factor file (from July 1963), **1,196** in the momentum file and in the file of ten momentum portfolios (from January 1927).
- Daily files start on 1 July 1926 (three factors), 1 July 1963 (five factors) and 3 November 1926 (momentum and the ten portfolios), and end on 31 August 2026.
- Trading days per month run from **17 to 27 before June 1952** and from **15 to 23 from 1953 on**, because the daily files carry Saturday trading days until **24 May 1952**, when the New York Stock Exchange stopped trading on Saturdays. No month has fewer than 15 trading days, the level below which a month is flagged (`TRADING_DAYS_FLAG_BELOW`).
- The samples, each counted from the factor's first month with a volatility-managed return (August 1926 for Mkt-RF, SMB and HML, January 1927 for Mom, August 1963 for RMW and CMA; Cederburg et al., Table 1 note, p. 99):
    - Moreira and Muir's sample, to April 2015: **1,065, 1,060 and 621** months, the N their Table I prints (p. 1619). The paper says only "1926 to 2015"; the end in April 2015 is inferred from these three counts, and this notebook confirms that the data hold a return for every one of those months.
    - Cederburg et al.'s sample, to December 2016: **1,085, 1,080 and 641** months (their Table 1 note, p. 99).
    - The months after Moreira and Muir's sample, May 2015 to August 2026: **136** months, the study's own question.
    - Barroso and Santa-Clara's sample: the paper's runs from March 1927 to December 2011 (Table 3, p. 116). French's daily file of the ten momentum portfolios starts on 3 November 1926, and the variance they scale by needs 126 trading days, which are first available at the end of April 1927. The first scaled month is therefore **May 1927**, two months after the paper's, and the sample holds **1,016** months.

## The constants, the published numbers and the loader

Every sample boundary, construction parameter and expectation lives in one module, `src/sq/constants.py`, with its source beside it; the notebooks read these names and never introduce numbers of their own. The published numbers of the three papers live in `src/sq/published.py`, each with its page and table. The loader is `src/sq/french_loader.py`. The cells below write the three modules into the runtime, so that this notebook runs from a blank Colab runtime; in the repository the files are the same text.

In [ ]:
import os
os.makedirs("src/sq", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
open("src/sq/__init__.py", "w").close()

In [ ]:
%%writefile src/sq/constants.py
"""
Pre-committed numbers for the study.

Every sample boundary, construction parameter, expectation and test setting is
defined here, one per name, with its source beside it. A notebook may read
these names; it may not introduce numbers of its own. The design was fixed on
8 November 2026 and restated against the three papers' printed tables on
10 November 2026, before any code that judges an expectation was written.

The three papers, with the journal's page numbers used throughout:

  MM    Moreira and Muir (2017), "Volatility-Managed Portfolios", Journal of
        Finance 72(4), 1611-1644.
  COWY Cederburg, O'Doherty, Wang and Yan (2020), "On the performance of
        volatility-managed portfolios", Journal of Financial Economics 138(1),
        95-117.
  BSC   Barroso and Santa-Clara (2015), "Momentum has its moments", Journal of
        Financial Economics 116(1), 111-120.

The Sharpe ratio test is Ledoit and Wolf (2008), "Robust performance
hypothesis testing with the Sharpe ratio", Journal of Empirical Finance 15(5),
850-859. The published numbers the expectations compare against are typed in
src/sq/published.py, never here.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":       "F-F_Research_Data_Factors_CSV.zip",             # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",       # the same, daily from 1926-07-01
    "factors5":       "F-F_Research_Data_5_Factors_2x3_CSV.zip",       # adds RMW and CMA, monthly from 1963-07
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # the same, daily from 1963-07-01
    "momentum":       "F-F_Momentum_Factor_CSV.zip",                   # Mom, monthly from 1927-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",             # Mom, daily from 1926-11-03
    "mom10":          "10_Portfolios_Prior_12_2_CSV.zip",              # ten portfolios formed on momentum, monthly from 1927-01
    "mom10_daily":    "10_Portfolios_Prior_12_2_Daily_CSV.zip",        # the same, daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# The six factors, by French's column names, and the file each comes from.
# Mkt-RF, SMB and HML come from the three-factor file, as in MM Table I and
# COWY section 2.1 (Fama and French 1993); RMW and CMA from the five-factor
# file (Fama and French 2015); Mom from the momentum file. RF is the
# three-factor file's one-month Treasury bill return.
FACTORS = ("Mkt-RF", "SMB", "HML", "Mom", "RMW", "CMA")
FACTOR_SOURCE = {
    "Mkt-RF": "factors3", "SMB": "factors3", "HML": "factors3",
    "Mom": "momentum", "RMW": "factors5", "CMA": "factors5",
}
RF_SOURCE = "factors3"

# WML, the momentum series of BSC: the top decile minus the bottom decile of
# the ten value-weighted portfolios formed on returns from month t-12 to t-2
# (BSC Appendix A, p. 119). Block 0 of both files holds the value-weighted returns.
WML_LONG = "Hi PRIOR"
WML_SHORT = "Lo PRIOR"
WML_BLOCK = 0
WML_BLOCK_TITLE_MUST_CONTAIN = "Value Weight"

# The end of the sample, fixed on 8 November 2026: August 2026 on any vintage.
# The loader cuts every file at this month, so that each notebook sees the same
# months on any later vintage; the vintage line and the SHA-256 in the
# provenance record say which revision of French's history a run used. A run on
# a later vintage, cut at August 2026, is a confirmation run: it is labelled by
# its vintage in the provenance record, it is expected to reproduce every
# verdict line and every printed decimal and not every digit, because French
# revises history, and it replaces no committed table without a dated decision.
SAMPLE_END = "2026-08"
SAMPLE_END_RULE_DATE = "2026-11-08"
CONFIRMATION_RUN = "a run on a later vintage, cut at August 2026, labelled by its vintage in the provenance record"

# French's daily files carry Saturday trading days until 24 May 1952. The number of
# trading days in a calendar month runs from 17 to 27 before June 1952 and from 15
# to 23 from 1953 on (counted on vintage 202609). Months with fewer trading days
# than the level below are flagged and reported; none is dropped.
LAST_SATURDAY_TRADING_DAY = "1952-05-24"
TRADING_DAYS_FLAG_BELOW = 15

PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

# ---------------------------------------------------------------------------
# 2. The samples
# ---------------------------------------------------------------------------

# The first month with a volatility-managed return, for each factor: the month
# after the first month of daily data, or the first month of the monthly file
# if that is later. These are COWY's start dates (Table 1 note, p. 99).
FACTOR_FIRST_MONTH = {
    "Mkt-RF": "1926-08", "SMB": "1926-08", "HML": "1926-08",
    "Mom": "1927-01", "RMW": "1963-08", "CMA": "1963-08",
}

# MM's sample. The paper says "1926 to 2015" (Table I note, p. 1619) and prints
# N = 1,065 for Mkt, SMB and HML, 1,060 for Mom and 621 for RMW and CMA. From
# the first months above, all three counts end in April 2015 exactly; an end in
# December 2015 would give 1,073, 1,068 and 629. The paper does not state the
# months, so the end is inferred from these counts; the computed N is reported
# beside the published N in every table that judges E1.
MM_SAMPLE_END = "2015-04"
MM_SAMPLE_END_IS_INFERRED = True
MM_EXPECTED_N = {"Mkt-RF": 1065, "SMB": 1065, "HML": 1065, "Mom": 1060, "RMW": 621, "CMA": 621}

# COWY's sample: the same first months to December 2016 (Table 1 note, p. 99).
COWY_SAMPLE_END = "2016-12"
COWY_EXPECTED_N = {"Mkt-RF": 1085, "SMB": 1085, "HML": 1085, "Mom": 1080, "RMW": 641, "CMA": 641}

# The months after MM's sample, the study's own question (E5).
AFTER_START = "2015-05"
AFTER_END = SAMPLE_END
AFTER_EXPECTED_MONTHS = 136

# BSC's sample is 1927:03 to 2011:12 (Table 3, p. 116). Their daily WML starts
# in August 1926 (Daniel and Moskowitz's data, Appendix A, p. 119); French's
# daily file of the ten momentum portfolios starts on 3 November 1926, so the
# 126 trading days the variance needs are first available at the end of April 1927
# and the first scaled month is May 1927, two months after the paper's.
BSC_PAPER_FIRST_MONTH = "1927-03"
BSC_FIRST_MONTH = "1927-05"
BSC_SAMPLE_END = "2011-12"
BSC_EXPECTED_MONTHS = 1016          # May 1927 to December 2011 inclusive
BSC_EXTENDED_END = SAMPLE_END       # the second period of scaled momentum: May 1927 to August 2026

# ---------------------------------------------------------------------------
# 3. The constructions
# ---------------------------------------------------------------------------

# The signal: last month's realised variance of the factor's daily returns.
# Each construction uses its own paper's definition.
#
# MM equation 2, p. 1616: each day's return minus the month's mean, squared and
# summed over the month. The equation is written for a 22-day month; the mean
# here is the month's own mean, the sum of its daily returns divided by its
# number of trading days.
RV_MM = "sum over the month's trading days of (daily return minus the month's mean daily return) squared"
# COWY equation 4, p. 98: 22 divided by the month's number of trading days J,
# times the sum of squared daily returns; no mean is removed.
RV_COWY_DAYS = 22
RV_COWY = "22 / J times the sum over the month's J trading days of the squared daily return"
# BSC equation 5, p. 115: 21 times the mean squared daily return over the last
# 126 trading days up to the month's last trading day, a monthly variance.
RV_BSC_TRADING_DAYS = 126
RV_BSC_DAYS_PER_MONTH = 21

# Which definition each comparison uses: a comparison with a paper's printed
# numbers uses that paper's definition. Fixed 10 November 2026.
RV_BY_COMPARISON = {
    "volatility-managed factor on MM's sample: E1, E4, E6 (Table IV, Table V)": "MM",
    "volatility-managed factor on COWY's sample: E2 (i), E6 (Table 1, Panel D)": "COWY",
    "real-time combination and real-time original: E2 (ii) to (iv), E5": "COWY",
    "volatility-managed factor after the sample, c fixed to April 2015: E5, no verdict": "MM",
    "scaled WML: E3": "BSC",
}

# The volatility-managed factor (MM equation 1, p. 1616): f_sigma_t = (c / RV_{t-1}) f_t,
# with c set so that the managed and the original factor have the same standard
# deviation over the sample the comparison runs on. c does not change a Sharpe
# ratio or an appraisal ratio (MM footnote 6, p. 1616; COWY p. 98).
#
# The real-time combination (COWY section 4.3, pp. 106-107, Table 5 Panel A):
# at the start of each month t after an initial training period of K months,
# on an expanding window of all months before t,
#   1. c_t gives the managed and the original factor the same variance over the window;
#   2. the means mu and the covariance Sigma of the managed and the original factor
#      are estimated over the window;
#   3. the weights are (x_sigma, x) = (1 / gamma) Sigma^-1 mu;
#   4. the position in the original factor is y_t = x_sigma c_t / RV_{t-1} + x, with |y_t| <= cap;
#   5. the month's return is y_t f_t.
# S2, the real-time original, is the same with the original factor alone:
# y_t = (1 / gamma) mu / sigma^2, under the same cap. S3, the ex post optimal
# combination, uses the moments of the whole out-of-sample period.
REALTIME_K = 120               # months; COWY p. 106, "K = 120 months"
REALTIME_GAMMA = 5.0           # risk aversion; COWY Table 5 note, p. 106
REALTIME_LEVERAGE_CAP = 5.0    # |y_t| <= 5; COWY Table 5 note, p. 106, and p. 107
REALTIME_WINDOW = "expanding"  # COWY p. 106

# Scaled momentum (BSC equation 6, p. 115): WML scaled by sigma_target / sigma_hat_{t-1},
# sigma_hat the square root of equation 5's variance, sigma_target an annualised
# volatility of 12 per cent (BSC p. 115), here as a monthly figure.
BSC_TARGET_VOL_ANNUAL = 0.12
BSC_TARGET_VOL_MONTHLY = BSC_TARGET_VOL_ANNUAL / 12 ** 0.5

# ---------------------------------------------------------------------------
# 4. Measures and tests (the decisions with defaults of 8 November 2026)
# ---------------------------------------------------------------------------

MONTHS_PER_YEAR = 12
# The regression f_sigma_t = alpha + beta f_t + e_t, monthly. Alpha is reported
# in per cent a year (times 12); the appraisal ratio is alpha / sd(e) times the
# square root of 12 (MM footnote 11, p. 1620); Sharpe ratios are annualised by
# the square root of 12.
ALPHA_ANNUALISE = MONTHS_PER_YEAR
SHARPE_ANNUALISE = MONTHS_PER_YEAR ** 0.5

# Standard errors. E1 is judged with heteroskedasticity-robust standard errors,
# the papers' own (MM Table I note, p. 1619; COWY Table 3 note, White 1980,
# p. 104). The Newey and West standard error at 12 lags is reported beside it
# and is the one E4 to E6 use. HML's published alpha, 1.97 with a standard
# error of 1.02, has a t-statistic of 1.93 against the 1.96 line, so the
# choice can move a factor across it.
SE_E1 = "HC0"                  # White (1980)
NEWEY_WEST_LAGS = 12
SIGNIFICANCE_LEVEL = 0.05

# The Sharpe ratio difference (Ledoit and Wolf 2008): the studentised circular
# block bootstrap on paired series.
BOOTSTRAP_BLOCK_MONTHS = 12
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_LEVEL = 0.95
BOOTSTRAP_SEED = 20261108      # the date of the design, as a number

# The size check, run before the test touches a factor: 1,000 simulated pairs
# with equal Sharpe ratios and the sample's length, Student-t shocks with five
# degrees of freedom, a common component with the market's correlation between
# the pair, no difference in means.
SIZE_CHECK_PAIRS = 1000
SIZE_CHECK_T_DOF = 5
SIZE_CHECK_SEED = 20261110

# Costs: one proportional cost per unit of one-way turnover, the turnover of a
# volatility-managed strategy in a month being the absolute change in its
# position, |w_t - w_{t-1}|. Factor portfolios are traded through liquid
# large-cap long-short books, for which 10 basis points per unit of notional
# traded is the order the literature on implementation costs reports (MM Table
# IV, p. 1625, takes 10 basis points from Frazzini, Israel and Moskowitz); the
# one sensitivity doubles it.
COST_PER_TURNOVER = 0.0010
COST_SENSITIVITY = 0.0020

# ---------------------------------------------------------------------------
# 5. The specifications, counted before the code runs
# ---------------------------------------------------------------------------

# A specification is one pair of return series compared on one sample. The
# list is written out so that nothing can be added after a run without a dated
# disclosure. Cost levels are sensitivities of the same specifications and are
# not counted again.
_SAMPLES = {
    "MM": "first month to April 2015",
    "COWY": "first month to December 2016",
    "AFTER": "May 2015 to August 2026",
}
SPECIFICATIONS = (
    tuple(("E1", f, "MM", "volatility-managed", "original") for f in FACTORS)
    + tuple(("E2 (i)", f, "COWY", "volatility-managed", "original") for f in FACTORS)
    + tuple(("E2 (ii, iii)", f, "COWY", "real-time combination S1", "real-time original S2") for f in FACTORS)
    + tuple(("E2 (iv)", f, "COWY", "real-time combination S1", "ex post optimal combination S3") for f in FACTORS)
    + tuple(("E5", f, "AFTER", "real-time combination S1", "real-time original S2") for f in FACTORS)
    + tuple(("E5, no verdict", f, "AFTER", "volatility-managed, c to April 2015", "original") for f in FACTORS)
    + (("E3", "WML", "BSC 1927-05 to 2011-12", "scaled WML", "WML"),
       ("E3, extended", "WML", "BSC 1927-05 to 2026-08", "scaled WML", "WML"))
)
SPECIFICATION_COUNT = 38
SPECIFICATION_COUNT_DATE = "2026-11-10"

# ---------------------------------------------------------------------------
# 6. The expectations, written before the code
# ---------------------------------------------------------------------------

# Each is a strict pass or fail, reported in the same voice whether met or
# not; no level moves after a run. The levels the code reads are the named
# constants; the text and the reason are what each notebook quotes.

E1_MIN_FACTORS = 5
E1_ALPHA_WITHIN_SE = 1.0
E1_APPRAISAL_BAND = 0.05
E1_MARKET_SHARPE_BAND = 0.05

E2_SHARPE_BAND = 0.05
E2_SIGN_MIN_FACTORS = 5
E2_ONLY_SIGNIFICANT = ("Mom",)
E2_S1_S3_BAND = 0.05

E3_SHARPE_BAND = 0.05
E3_DRAWDOWN_BAND = 0.05        # five percentage points of drawdown

E4_SIZE_RANGE = (0.035, 0.065)
E4_MIN_FACTORS = 5

E5_MAX_FACTORS_ABOVE = 2

E6_COST = COST_PER_TURNOVER
E6_MIN_NW_T = 2.0
E6_MARKET_PERCENTILES = (50, 75, 90, 99)
E6_PERCENTILE_BAND = 0.15      # P50, P75 and P90
E6_P99_BAND = 1.0

EXPECTATIONS = {
    "E1": {
        "date": "2026-11-10",
        "text": (
            "On MM's sample (each factor's first month to April 2015), for at least five of the six factors: "
            "the alpha's sign and its 5 per cent significance category, with heteroskedasticity-robust (White) "
            "standard errors, match Table I (p. 1619); the alpha lies within one published standard error of the "
            "published alpha; and the appraisal ratio, computed from Table I's alpha and RMSE with footnote 11's "
            "annualisation (p. 1620), lies within 0.05 of the paper's figure, the market's 0.33 from the table, with "
            "the text's 0.34 (p. 1620) and 0.33 (p. 1612) noted beside it. For the market, the volatility-managed "
            "Sharpe ratio also lies within 0.05 of 0.52 (Table V, Panel A, p. 1626), and the market counts as "
            "matching only if it does. Every factor's comparisons are reported, whatever the count, with the "
            "computed N beside the published N."
        ),
        "reason": (
            "The replication criterion fixed on 6 September 2026: sign and significance, one standard error, and a "
            "band of 0.05. Its Sharpe ratio leg is the appraisal ratio for every factor because the paper prints no "
            "Sharpe ratio for any factor but the market. French revises history, so exact equality is not expected; "
            "five of six allows one factor to miss on a revised series, and the miss is reported, not chased."
        ),
    },
    "E2": {
        "date": "2026-11-10",
        "text": (
            "On COWY's sample (August 1926, January 1927 or August 1963 to December 2016), for each of the six "
            "factors: (i) the original and the volatility-managed Sharpe ratios lie within 0.05 of their Table 1 "
            "(p. 99); (ii) the real-time combination's Sharpe ratio S1 and the real-time original's S2 lie within "
            "0.05 of Table 5, Panel A (p. 106); (iii) S1 minus S2 has the published sign for at least five of the "
            "six, and the only difference significant at 5 per cent by the Jobson and Korkie test is momentum's, "
            "as published; (iv) S1 minus S3, the ex post optimal combination, lies within 0.05 of the published "
            "figure for all six. All twenty-four comparisons are reported, whatever the count."
        ),
        "reason": (
            "The critique's baseline is a real-time combination of the volatility-managed and the original factor "
            "(section 4.3, pp. 106-107): the scaling constant alone does not move a Sharpe ratio (MM footnote 6, "
            "p. 1616; COWY p. 98), and the weights of the combination are what an investor cannot know in advance."
        ),
    },
    "E3": {
        "date": "2026-11-10",
        "text": (
            "On May 1927 to December 2011, the Sharpe ratios of unscaled and scaled WML lie within 0.05 of 0.53 and "
            "0.97 (BSC Table 3, p. 116), and the maximum drawdowns lie within five percentage points of 96.69 and "
            "45.20 per cent (p. 112). The first scaled month is two months after the paper's 1927:03 because "
            "French's daily file of the ten momentum portfolios starts on 3 November 1926."
        ),
        "reason": (
            "The paper's two headline results, on its own momentum series (Appendix A, p. 119); the Sharpe band is "
            "the one used throughout the study."
        ),
    },
    "E4": {
        "date": "2026-11-08",
        "text": (
            "The size check rejects between 3.5 and 6.5 per cent of the 1,000 simulated null pairs at the 5 per cent "
            "level; and on MM's sample the bootstrap's verdict on the Sharpe ratio difference agrees in sign and in "
            "significance with the Newey and West t-statistic on the alpha for at least five of the six factors."
        ),
        "reason": (
            "A test that is wrong about its own size is not used; the band is two standard errors of a proportion "
            "of 0.05 over 1,000 draws. The alpha and the Sharpe ratio difference are two readings of the same gain "
            "and should agree."
        ),
    },
    "E5": {
        "date": "2026-11-10",
        "text": (
            "On the months after MM's sample, May 2015 to August 2026 (136 months), with the paper's c fixed on the "
            "data to April 2015 and the real-time combination carried forward from its expanding window: for each "
            "factor, the Sharpe ratio of the real-time combination against the real-time original S2, as in COWY "
            "Table 5, and of the volatility-managed factor against the original factor itself, each difference with "
            "its Ledoit and Wolf bootstrap interval. On at most two of the six factors the real-time combination's "
            "Sharpe ratio exceeds S2's by more than the interval allows, and the median across the six of the Sharpe "
            "difference lies within plus or minus the median of the six intervals' half-widths. The same two counts "
            "are reported for the volatility-managed factor, in no verdict."
        ),
        "reason": (
            "The study's own question, with the prior the critique gives: gains an investor could not have set in "
            "real time should show little in months neither paper could have known. If more than two factors clear "
            "the interval, the expectation fails and the finding is reported as such."
        ),
    },
    "E6": {
        "date": "2026-11-10",
        "text": (
            "At 10 basis points per unit of turnover, the market's volatility-managed alpha over MM's sample stays "
            "positive and more than two Newey and West standard errors from zero (the paper: 3.98 per cent a year at "
            "10 basis points, an average absolute change in weight of 0.73 and a break-even cost of 56 basis points, "
            "Table IV, p. 1625). On MM's sample, the market's P50, P75 and P90 of the leverage lie within 0.15 of "
            "0.93, 1.59 and 2.64, and its P99 within 1.0 of 6.39 (Table V, Panel A, p. 1626); on COWY's sample, each "
            "factor's P50 lies within 0.15 of their Table 1, Panel D (p. 99), and its P99 within 1.0 of theirs (6.47, "
            "5.07, 5.89, 8.64, 5.02, 4.56)."
        ),
        "reason": (
            "MM report that the market strategy's gain survives moderate costs (Table IV) and print the distribution "
            "of its leverage (Table V); COWY print the 1st, 50th and 99th percentiles for every factor. The "
            "expectation says whether both hold on this vintage."
        ),
    },
}

In [ ]:
%%writefile src/sq/published.py
"""
The published numbers of the three papers the study replicates, transcribed
from the journal PDFs, with the page and the table of every number.

The comparison tables in the notebooks are built from this module, never typed
into a notebook. tests/test_published.py re-reads every number from the
journal PDFs when they are present on the machine (they are copyrighted and
live outside the repository) and fails on any digit that differs.

Units are the papers' own: returns, alphas and standard deviations in per cent
a year, Sharpe and appraisal ratios annualised, leverage as a multiple of the
original factor. The keys are French's column names for the six factors
(MM print Mkt, SMB, HML, Mom, RMW, CMA; COWY print MKT, SMB, HML, MOM, RMW, CMA).
"""
from __future__ import annotations

import math

FACTORS = ("Mkt-RF", "SMB", "HML", "Mom", "RMW", "CMA")

# ---------------------------------------------------------------------------
# Moreira and Muir (2017), Journal of Finance 72(4), 1611-1644
# ---------------------------------------------------------------------------

# Table I, Panel A, p. 1619: f_sigma = alpha + beta f + e, univariate, monthly
# data with alpha and RMSE multiplied by 12; standard errors adjust for
# heteroskedasticity. The table's first six of ten columns.
MM_TABLE_I = {
    #          beta  (se)   alpha  (se)     N     R2    RMSE
    "Mkt-RF": (0.61, 0.05,  4.86, 1.56, 1065, 0.37, 51.39),
    "SMB":    (0.62, 0.08, -0.58, 0.91, 1065, 0.38, 30.44),
    "HML":    (0.57, 0.07,  1.97, 1.02, 1065, 0.32, 34.92),
    "Mom":    (0.47, 0.07, 12.51, 1.71, 1060, 0.22, 50.37),
    "RMW":    (0.62, 0.08,  2.44, 0.83,  621, 0.38, 20.16),
    "CMA":    (0.68, 0.05,  0.38, 0.67,  621, 0.46, 17.55),
}
MM_TABLE_I_FIELDS = ("beta", "beta_se", "alpha", "alpha_se", "N", "R2", "RMSE")

# Appraisal ratios printed in the text: the market's 0.34 and HML's 0.20 and
# profitability's 0.41 (p. 1620), and the market's 0.33 in the introduction
# (p. 1612). The figures E1 compares against are computed from Table I below.
MM_TEXT_APPRAISAL_P1620 = {"Mkt-RF": 0.34, "HML": 0.20, "RMW": 0.41}
MM_TEXT_APPRAISAL_P1612 = {"Mkt-RF": 0.33}

# Table IV, p. 1625: the market, "Realized variance" row. Average absolute
# change in monthly weight, expected return and alpha before costs (per cent a
# year), alpha after 1, 10 and 14 basis points of trading cost, and the
# break-even cost in basis points.
MM_TABLE_IV_MARKET = {
    "mean_abs_change_in_weight": 0.73,
    "expected_return": 9.47,
    "alpha": 4.86,
    "alpha_1bp": 4.77,
    "alpha_10bp": 3.98,
    "alpha_14bp": 3.63,
    "break_even_bp": 56,
}

# Table V, Panel A, p. 1626: the market, "Realized variance" row. Alpha and its
# standard error, Sharpe ratio, appraisal ratio, and percentiles of the weight
# c / RV.
MM_TABLE_V_MARKET = {
    "alpha": 4.86, "alpha_se": 1.56, "sharpe": 0.52, "appraisal": 0.34,
    "P50": 0.93, "P75": 1.59, "P90": 2.64, "P99": 6.39,
}


def mm_appraisal_from_table_i(factor: str) -> float:
    """alpha / RMSE times the square root of 12, from Table I (footnote 11, p. 1620)."""
    beta, beta_se, alpha, alpha_se, n, r2, rmse = MM_TABLE_I[factor]
    return alpha / rmse * math.sqrt(12)


# ---------------------------------------------------------------------------
# Cederburg, O'Doherty, Wang and Yan (2020), Journal of Financial Economics
# 138(1), 95-117. Their sample ends in December 2016 (Table 1 note, p. 99).
# ---------------------------------------------------------------------------

# Table 1, p. 99. Panel A, original factors, and Panel B, volatility-managed
# factors: mean, standard deviation (per cent a year) and Sharpe ratio.
COWY_TABLE_1_ORIGINAL = {
    #          mean    sd     SR
    "Mkt-RF": (7.80, 18.61, 0.42),
    "SMB":    (2.57, 11.12, 0.23),
    "HML":    (4.84, 12.14, 0.40),
    "Mom":    (7.94, 16.39, 0.48),
    "RMW":    (2.92,  7.71, 0.38),
    "CMA":    (3.72,  6.97, 0.53),
}
COWY_TABLE_1_MANAGED = {
    "Mkt-RF": (9.55, 18.61, 0.51),
    "SMB":    (0.86, 11.12, 0.08),
    "HML":    (4.64, 12.14, 0.38),
    "Mom":    (16.17, 16.39, 0.99),
    "RMW":    (3.94,  7.71, 0.51),
    "CMA":    (2.79,  6.97, 0.40),
}
# Panel C: Sharpe ratio difference, managed minus original, and the Jobson and
# Korkie p-value. Panel D: correlation with the original factor and the 1st,
# 50th and 99th percentiles of the weight c* / RV.
COWY_TABLE_1_DIFFERENCE = {
    "Mkt-RF": (0.09, 0.30), "SMB": (-0.15, 0.09), "HML": (-0.02, 0.86),
    "Mom": (0.50, 0.00), "RMW": (0.13, 0.29), "CMA": (-0.13, 0.23),
}
COWY_TABLE_1_PANEL_D = {
    #          corr   P01   P50   P99
    "Mkt-RF": (0.63, 0.04, 0.96, 6.47),
    "SMB":    (0.63, 0.03, 0.81, 5.07),
    "HML":    (0.57, 0.04, 1.02, 5.89),
    "Mom":    (0.48, 0.04, 1.01, 8.64),
    "RMW":    (0.59, 0.04, 1.11, 5.02),
    "CMA":    (0.68, 0.06, 0.97, 4.56),
}

# Table 3, Panel A.1, p. 104: the full-sample spanning regression. Alpha (per
# cent a year) with its t-statistic from White (1980) standard errors, beta with
# its t-statistic, adjusted R2, appraisal ratio; Panel A.2, the scaling constant c*.
COWY_TABLE_3 = {
    #          alpha  (t)    beta  (t)     R2    AR     c*
    "Mkt-RF": (4.63,  3.08, 0.63, 11.32, 0.40,  0.32, 10.33),
    "SMB":    (-0.76, -0.87, 0.63,  7.75, 0.40, -0.09,  2.63),
    "HML":    (1.87,  1.88, 0.57,  7.65, 0.33,  0.19,  2.95),
    "Mom":    (12.39, 7.31, 0.48,  7.13, 0.23,  0.86,  4.60),
    "RMW":    (2.23,  2.57, 0.59,  7.10, 0.34,  0.36,  1.48),
    "CMA":    (0.26,  0.39, 0.68, 13.82, 0.46,  0.05,  1.53),
}
COWY_TABLE_3_FIELDS = ("alpha", "alpha_t", "beta", "beta_t", "R2", "AR", "c")

# Table 5, Panel A, p. 106: the real-time strategies, K = 120, expanding
# window, gamma = 5, leverage at most 5; Sharpe ratios over the out-of-sample
# period, p-values of the differences in the second tuple (Jobson and Korkie).
COWY_TABLE_5 = {
    #          S1    S2    S1-S2  S3    S1-S3
    "Mkt-RF": (0.42, 0.46, -0.04, 0.53, -0.11),
    "SMB":    (0.14, 0.19, -0.06, 0.26, -0.12),
    "HML":    (0.38, 0.43, -0.06, 0.50, -0.12),
    "Mom":    (0.92, 0.49,  0.44, 0.99, -0.07),
    "RMW":    (0.44, 0.31,  0.13, 0.58, -0.14),
    "CMA":    (0.52, 0.56, -0.03, 0.64, -0.11),
}
COWY_TABLE_5_P = {
    #          S1-S2  S1-S3
    "Mkt-RF": (0.64, 0.01),
    "SMB":    (0.37, 0.14),
    "HML":    (0.41, 0.08),
    "Mom":    (0.00, 0.07),
    "RMW":    (0.53, 0.37),
    "CMA":    (0.20, 0.00),
}

# ---------------------------------------------------------------------------
# Barroso and Santa-Clara (2015), Journal of Financial Economics 116(1), 111-120.
# Sample 1927:03 to 2011:12.
# ---------------------------------------------------------------------------

BSC_FIELDS = ("max", "min", "mean", "sd", "kurtosis", "skewness", "sharpe")
# Table 1, p. 113, and Table 3, p. 116: monthly returns in per cent, mean and
# standard deviation annualised, excess kurtosis, skewness, annualised Sharpe ratio.
BSC_TABLE_1_WML = (26.18, -78.96, 14.46, 27.53, 18.24, -2.47, 0.53)
BSC_TABLE_3_WML = (26.18, -78.96, 14.46, 27.53, 18.24, -2.47, 0.53)
BSC_TABLE_3_WML_SCALED = (21.95, -28.40, 16.50, 16.95, 2.68, -0.42, 0.97)
BSC_TABLE_3_INFORMATION_RATIO = 0.78
# p. 112: maximum drawdown in per cent, unscaled and scaled.
BSC_MAX_DRAWDOWN = {"WML": -96.69, "WML scaled": -45.20}


def table(name: str):
    """One published table as a pandas DataFrame, factors as rows."""
    import pandas as pd
    if name == "mm_table_i":
        df = pd.DataFrame(MM_TABLE_I, index=MM_TABLE_I_FIELDS).T
        df["appraisal"] = [mm_appraisal_from_table_i(f) for f in df.index]
        return df
    if name == "cowy_table_1":
        o = pd.DataFrame(COWY_TABLE_1_ORIGINAL, index=["mean", "sd", "sharpe"]).T
        m = pd.DataFrame(COWY_TABLE_1_MANAGED, index=["mean", "sd", "sharpe"]).T
        d = pd.DataFrame(COWY_TABLE_1_DIFFERENCE, index=["difference", "p"]).T
        q = pd.DataFrame(COWY_TABLE_1_PANEL_D, index=["corr", "P01", "P50", "P99"]).T
        return pd.concat({"original": o, "managed": m, "difference": d, "weights": q}, axis=1)
    if name == "cowy_table_3":
        return pd.DataFrame(COWY_TABLE_3, index=COWY_TABLE_3_FIELDS).T
    if name == "cowy_table_5":
        s = pd.DataFrame(COWY_TABLE_5, index=["S1", "S2", "S1-S2", "S3", "S1-S3"]).T
        p = pd.DataFrame(COWY_TABLE_5_P, index=["p S1-S2", "p S1-S3"]).T
        return pd.concat([s, p], axis=1)
    if name == "bsc_table_3":
        return pd.DataFrame({"WML": BSC_TABLE_3_WML, "WML scaled": BSC_TABLE_3_WML_SCALED}, index=BSC_FIELDS).T
    raise KeyError(name)

In [ ]:
%%writefile src/sq/french_loader.py
"""
Ken French data library: download, record, parse, assemble.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with one or more blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: eight digits
for daily data (19260701), six for monthly (192607), four for annual (1927).
Missing values are coded -99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a Period (day, month
or year), in decimal returns, with the missing codes replaced by NaN. It also
writes a provenance record for every file it downloads: the URL, the download
time, the SHA-256 of the zip, and the CRSP vintage line French prints at the
top of each CSV. A result is only as reproducible as the vintage it was
computed on, and French revises history, so the record is part of every output.

Two facts about the daily files matter for a monthly variance built from them.
They carry Saturday trading days until 24 May 1952, when the New York Stock
Exchange stopped trading on Saturdays. The number of trading days in a calendar
month therefore runs from 17 to 27 before June 1952 and from 15 to 23 from
1953 on (counted on vintage 202609). The variance definitions in constants.py
say how each paper treats the number of trading days.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{8}|\d{6}|\d{4})\s*,")   # daily YYYYMMDD, monthly YYYYMM, annual YYYY


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "daily" | "monthly" |
    "annual", "frame": DataFrame} where the frame is indexed by Period (day,
    month or year) and holds floats in the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 8:
                idx = pd.PeriodIndex(pd.to_datetime(raw.index.astype(str), format="%Y%m%d"), freq="D")
                freq = "daily"
            elif date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One block of one file, in decimal returns with missing codes as NaN.

    The name dates from the monthly files; the function serves the daily files
    in the same way (the block's index is then a daily Period).

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = cut_to_sample_end(chosen["frame"].copy())
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def cut_to_sample_end(frame: pd.DataFrame, end: str | None = None) -> pd.DataFrame:
    """The frame up to the end of the sample (constants.SAMPLE_END): a monthly block to that
    month, a daily block to the last day of that month, an annual block to that year. A later vintage of French's
    files carries months after the end, which the study does not use; the vintage in the provenance record still says
    which revision of history the months before the end came from. With no end set, the frame is returned as it is."""
    end = C.SAMPLE_END if end is None else end
    if end is None or not isinstance(frame.index, pd.PeriodIndex) or len(frame) == 0:
        return frame
    end_m = pd.Period(end, "M")
    freq = frame.index.freqstr or ""
    if freq.startswith("M"):
        limit = end_m
    elif freq.startswith("D"):
        limit = end_m.asfreq("D", "end")
    else:
        limit = end_m.asfreq(frame.index.freq, "end")
    return frame.loc[:limit]


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# The study's series, assembled from the files above
# ---------------------------------------------------------------------------

def _suffix(freq: str) -> str:
    if freq not in ("monthly", "daily"):
        raise ValueError(f"freq is 'monthly' or 'daily', not {freq!r}")
    return "" if freq == "monthly" else "_daily"


def factors(freq: str = "monthly", cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """The six factors of constants.FACTORS and the risk-free rate, in decimal returns, cut at the sample's end.

    Each factor comes from the file constants.FACTOR_SOURCE names; the columns are aligned on the union of the
    dates, so a factor whose file starts later (momentum, RMW and CMA) is NaN before its first date.
    """
    sfx = _suffix(freq)
    cols = {}
    frames = {}
    for f in C.FACTORS + ("RF",):
        key = (C.FACTOR_SOURCE[f] if f != "RF" else C.RF_SOURCE) + sfx
        if key not in frames:
            frames[key] = load_monthly(key, 0, cache_dir)
        cols[f] = frames[key][f]
    out = pd.DataFrame(cols)
    out.attrs["provenance"] = [frames[k].attrs["provenance"] for k in sorted(frames)]
    return out


def wml(freq: str = "monthly", cache_dir: str | Path = "data/french") -> pd.Series:
    """WML: the top decile minus the bottom decile of the ten value-weighted portfolios formed on momentum,
    in decimal returns, cut at the sample's end (Barroso and Santa-Clara, Appendix A)."""
    key = "mom10" + _suffix(freq)
    frame = load_monthly(key, C.WML_BLOCK, cache_dir)
    if C.WML_BLOCK_TITLE_MUST_CONTAIN not in frame.attrs["title"]:
        raise ValueError(f"block {C.WML_BLOCK} of {key} is {frame.attrs['title']!r}, not the value-weighted returns")
    out = (frame[C.WML_LONG] - frame[C.WML_SHORT]).rename("WML")
    out.attrs["provenance"] = frame.attrs["provenance"]
    return out


def trading_days_per_month(daily: pd.DataFrame | pd.Series) -> pd.Series:
    """The number of daily rows in each calendar month, indexed by month."""
    idx = daily.index
    months = idx.asfreq("M") if isinstance(idx, pd.PeriodIndex) else pd.PeriodIndex(idx, freq="M")
    return pd.Series(1, index=months).groupby(level=0).sum().rename("trading days")

## Download and record

The loader downloads each file once into `data/french/` and keeps it there; a rerun reads the cached zip and rebuilds its record from the bytes on disk, so the checksum always describes the file actually used. The record goes to `outputs/provenance_french.json`, which every later notebook reads.

In [ ]:
import sys
sys.path.insert(0, "src")
import pandas as pd
from sq import constants as C, published as P, french_loader as fl

records = [fl.download(key) for key in C.FRENCH_FILES]
fl.write_provenance(records)
prov = pd.DataFrame(records)[["key", "filename", "vintage_line", "zip_sha256", "zip_bytes", "download_time_local"]]
prov["zip_sha256"] = prov["zip_sha256"].str[:16]
print(prov.to_string(index=False))

vintages = {r["vintage_line"].split("the ")[-1].split(" ")[0] for r in records}
print("vintage(s) in this run:", vintages)
VINTAGE = sorted(vintages)[-1]

## What each file holds, after the cut at August 2026

The table shows each file's first block (the returns), its first and last date and its number of rows. The checks below the table stop the notebook if a count differs from the expectation.

In [ ]:
rows = []
for key in C.FRENCH_FILES:
    block = fl.load_monthly(key, 0)
    rows.append({"file": key, "block": block.attrs["title"], "rows": len(block),
                 "first": str(block.index[0]), "last": str(block.index[-1]), "columns": ", ".join(block.columns)})
files = pd.DataFrame(rows)
print(files.to_string(index=False))

n = files.set_index("file")["rows"]
assert n["factors3"] == 1202 and n["factors5"] == 758 and n["momentum"] == 1196 and n["mom10"] == 1196, n
first = files.set_index("file")["first"]
assert first["factors3_daily"] == "1926-07-01" and first["factors5_daily"] == "1963-07-01", first
assert first["momentum_daily"] == "1926-11-03" and first["mom10_daily"] == "1926-11-03", first
assert (files["last"].str[:7] == C.SAMPLE_END).all(), files["last"]

## The study's series

`fl.factors` puts the six factors and the risk-free rate side by side, in decimal returns (1.5 per cent is 0.015), each from the file the constants name: Mkt-RF, SMB and HML from the three-factor file, as both papers use them, RMW and CMA from the five-factor file, Mom from the momentum file. A factor whose file starts later is empty (NaN) before its first date. `fl.wml` takes the value-weighted block of the ten momentum portfolios and subtracts the bottom decile from the top one.

In [ ]:
monthly = fl.factors("monthly")
daily = fl.factors("daily")
wml_m = fl.wml("monthly")
wml_d = fl.wml("daily")

summary = pd.DataFrame({
    "first month": {f: str(monthly[f].first_valid_index()) for f in C.FACTORS},
    "months": monthly[list(C.FACTORS)].count(),
    "first trading day": {f: str(daily[f].first_valid_index()) for f in C.FACTORS},
    "trading days": daily[list(C.FACTORS)].count(),
})
summary.loc["WML"] = [str(wml_m.first_valid_index()), wml_m.count(), str(wml_d.first_valid_index()), wml_d.count()]
print(summary.to_string())

## Trading days per month

The realised variance of notebook 02 is built from the trading days of one month, so the number of trading days matters. Each paper treats it in its own way: Moreira and Muir sum over the month's trading days (their equation 2, p. 1616), Cederburg et al. rescale every month to 22 trading days (their equation 4, p. 98), and Barroso and Santa-Clara use the last 126 trading days whatever the month (their equation 5, p. 115). The counts below come from the market's daily series; the cell checks that every other daily file has the same trading days from its own first trading day on (momentum's first month, November 1926, starts on the 3rd).

In [ ]:
days = fl.trading_days_per_month(daily["Mkt-RF"].dropna())
saturdays = daily.index[daily.index.to_timestamp().dayofweek == 5]
before = days.loc[:"1952-05"]
after = days.loc["1953-01":]

print("trading days per month before June 1952:", before.min(), "to", before.max())
print("trading days per month from 1953 on:    ", after.min(), "to", after.max())
print("Saturday trading days:", len(saturdays), "the last on", saturdays.max())
print("months with the fewest trading days from 1953 on:")
print(after.nsmallest(3).to_string())

flagged = {}
for f in C.FACTORS:
    s = fl.trading_days_per_month(daily[f].dropna())
    # every daily file has the same trading days as the market's from its own first trading day on
    assert s.equals(days.loc[s.index[0]:s.index[-1]]) or s.iloc[1:].equals(days.loc[s.index[1]:s.index[-1]]), f
    flagged[f] = list(s[s < C.TRADING_DAYS_FLAG_BELOW].index.astype(str))
print("months with fewer than", C.TRADING_DAYS_FLAG_BELOW, "trading days:", flagged)

assert (before.min(), before.max(), after.min(), after.max()) == (17, 27, 15, 23)
assert str(saturdays.max()) == C.LAST_SATURDAY_TRADING_DAY
assert all(len(v) == 0 for v in flagged.values())
days.to_frame().to_csv(f"outputs/trading_days_per_month_{VINTAGE}.csv")

## The four samples

For each factor and each sample the cell counts the months that hold both a monthly return and at least one trading day in the month before, which is what a volatility-managed return needs. The published N of Moreira and Muir's Table I sits beside the count for their sample.

In [ ]:
def months_with_signal(f, start, end):
    m = monthly[f].loc[start:end].dropna()
    prior = set(fl.trading_days_per_month(daily[f].dropna()).index + 1)   # months whose previous month has trading days
    return sum(1 for p in m.index if p in prior)

rows = []
for f in C.FACTORS:
    first = C.FACTOR_FIRST_MONTH[f]
    rows.append({
        "factor": f, "first month": first,
        "MM to 2015-04": months_with_signal(f, first, C.MM_SAMPLE_END),
        "MM published N": P.MM_TABLE_I[f][4],
        "COWY to 2016-12": months_with_signal(f, first, C.COWY_SAMPLE_END),
        "after 2015-05 to 2026-08": months_with_signal(f, C.AFTER_START, C.AFTER_END),
    })
samples = pd.DataFrame(rows).set_index("factor")
print(samples.to_string())

assert (samples["MM to 2015-04"] == samples["MM published N"]).all()
assert (samples["COWY to 2016-12"] == pd.Series(C.COWY_EXPECTED_N)).all()
assert (samples["after 2015-05 to 2026-08"] == C.AFTER_EXPECTED_MONTHS).all()
for f in C.FACTORS:
    assert str(monthly[f].first_valid_index()) <= C.FACTOR_FIRST_MONTH[f], f

## Barroso and Santa-Clara's first scaled month

Their variance for month t uses the 126 trading days up to the last trading day of month t-1. The cell finds the first month-end by which WML's daily series has 126 trading days; the first scaled month is the month after it.

In [ ]:
wml_days = fl.trading_days_per_month(wml_d.dropna()).cumsum()
ready = wml_days[wml_days >= C.RV_BSC_TRADING_DAYS].index[0]
first_scaled = ready + 1
bsc_months = wml_m.loc[str(first_scaled):C.BSC_SAMPLE_END].dropna()
print("126 trading days first available at the end of", ready, "; first scaled month", first_scaled)
print("months from", first_scaled, "to", C.BSC_SAMPLE_END, ":", len(bsc_months))

assert str(first_scaled) == C.BSC_FIRST_MONTH and len(bsc_months) == C.BSC_EXPECTED_MONTHS

## The counts, written down

The counts go to `outputs/` with the vintage in each file name, beside the provenance record. The tables the write-up cites are copied from `outputs/` into `results/<vintage>/` and committed, so that every number resolves to a file at a named commit.

In [ ]:
files.to_csv(f"outputs/files_{VINTAGE}.csv", index=False)
summary.to_csv(f"outputs/series_{VINTAGE}.csv")
samples.to_csv(f"outputs/samples_{VINTAGE}.csv")
print(sorted(os.listdir("outputs")))

## What this notebook established, and what could be wrong

| Check | Expected | Where it comes from |
|---|---|---|
| Monthly rows to August 2026 | 1,202, 758, 1,196, 1,196 | the files' first months and the sample's end |
| Trading days per month | 17 to 27 before June 1952, 15 to 23 from 1953 | Saturday trading until 24 May 1952 |
| Months with fewer than 15 trading days | none | `TRADING_DAYS_FLAG_BELOW` |
| Moreira and Muir's sample to April 2015 | 1,065, 1,060, 621 | their Table I, p. 1619 |
| Cederburg et al.'s sample to December 2016 | 1,085, 1,080, 641 | their Table 1 note, p. 99 |
| Months after the sample | 136 | May 2015 to August 2026 |
| Barroso and Santa-Clara's sample | May 1927 to December 2011, 1,016 months | 126 trading days from 3 November 1926 |

Every assert in the notebook passed if the notebook ran to this cell.

What could be wrong:

- **The end of Moreira and Muir's sample is inferred.** April 2015 reproduces all three published N exactly; a start eight months later for every factor with an end in December 2015 reproduces them too. The inference is written in the constants with the three counts, and notebook 03 reports the computed N beside the published N.
- **The vintage differs from the papers'.** Moreira and Muir downloaded French's files around 2015 and Cederburg et al. around 2017; French has revised history since, so a computed number can miss a published one with the code right. The provenance record says which vintage every later number comes from.
- **WML starts two months later than the paper's.** The difference is two of the paper's 1,018 months and is disclosed wherever E3 is judged.